# Task 2.2 — Implementation of the Poisoning Attack
**Paper:** Poisoning Attacks against Support Vector Machines  
**Authors:** Battista Biggio, Blaine Nelson, Pavel Laskov (ICML 2012)  
**Roll Number:** 230110  
**Random Seed:** 42

---
## Contribution Being Reproduced

We reproduce the **gradient-based single-point poisoning attack** described in **Algorithm 1** of the paper.
This is the primary contribution: crafting a single malicious training point $x_c$ that, when injected into
the training set, maximally degrades the SVM's classification accuracy on held-out data.

**Evaluation metric:** Classification error rate on the test set (fraction of misclassified samples),
which is the same metric used in the paper's experiments (Figures 1–3).

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

We import standard scientific Python libraries (numpy, matplotlib, scikit-learn). No GPU or specialised
hardware is required — all computation runs on CPU.

In [2]:
# ── Data Loading and Preprocessing (same as Task 2.1) ──
data = load_breast_cancer()
X, y = data.data, data.target
y = 2 * y - 1  # Convert to +1/-1

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.35, random_state=RANDOM_SEED, stratify=y)
X_train, X_rem, y_train, y_rem = train_test_split(X_temp, y_temp, train_size=100, random_state=RANDOM_SEED)
X_val = X_rem[:100]
y_val = y_rem[:100]

C_SVM = 1.0  # Regularisation parameter, same as paper's Section 3.1

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"SVM kernel: linear, C = {C_SVM}")

Train: (100, 30), Val: (100, 30), Test: (200, 30)
SVM kernel: linear, C = 1.0


Data is loaded and split identically to Task 2.1. The regularisation parameter $C = 1$ matches the
paper's experimental setup (Section 3.1: *"The regularization parameter C was set to 1"*).

In [3]:
# ── Step 1: Train the clean (unpoisoned) baseline SVM ──
svm_clean = SVC(kernel='linear', C=C_SVM)
svm_clean.fit(X_train, y_train)

clean_test_err = 1.0 - accuracy_score(y_test, svm_clean.predict(X_test))
clean_val_err = 1.0 - accuracy_score(y_val, svm_clean.predict(X_val))

print(f"Clean SVM — Test error: {clean_test_err:.4f}, Val error: {clean_val_err:.4f}")
print(f"Number of support vectors: {len(svm_clean.support_)}")

Clean SVM — Test error: 0.0400, Val error: 0.0500
Number of support vectors: 32


The clean SVM serves as our baseline. Its test error rate will be compared against the poisoned SVM
to measure the attack's effectiveness. This corresponds to the initial state before any attack points
are injected (iteration 0 in the paper's Figure 1).

In [4]:
def compute_poisoning_gradient_linear(X_tr, y_tr, X_val, y_val, x_c, y_c, C_svm):
    """
    Compute the gradient of the validation hinge loss with respect to the
    attack point x_c, for a linear kernel SVM.
    
    This implements Equation (10) from Biggio et al. (2012):
    
        dL/du = sum_k [ M_k * dQ_sc/du + dQ_kc/du ] * alpha_c
    
    where M_k is defined using the inverse of Q_ss (the kernel matrix 
    restricted to margin support vectors) and the upsilon/zeta terms from
    the Sherman-Morrison-Woodbury decomposition (Equations 8-9).
    
    For the linear kernel, dK(x_i, x_c)/du = x_i (Section 2.2).
    
    Parameters
    ----------
    X_tr : array (n, d) — training features
    y_tr : array (n,)   — training labels (+1/-1)
    X_val : array (m, d) — validation features  
    y_val : array (m,)   — validation labels
    x_c : array (d,)     — current attack point
    y_c : int            — attack point label (+1 or -1)
    C_svm : float        — SVM regularisation parameter
    
    Returns
    -------
    gradient : array (d,) — dL/dx_c
    svm : trained SVC object
    """
    n_tr = len(X_tr)
    d = X_tr.shape[1]
    
    # Train SVM on augmented dataset D_tr ∪ {x_c, y_c}
    X_aug = np.vstack([X_tr, x_c.reshape(1, -1)])
    y_aug = np.append(y_tr, y_c)
    
    svm = SVC(kernel='linear', C=C_svm)
    svm.fit(X_aug, y_aug)
    
    # Extract dual variables alpha (scikit-learn stores alpha * y in dual_coef_)
    alpha_full = np.zeros(len(y_aug))
    sv_indices = svm.support_
    alpha_full[sv_indices] = np.abs(svm.dual_coef_[0])
    
    b = svm.intercept_[0]
    alpha_c = alpha_full[-1]  # dual variable for the attack point
    
    # If x_c is a reserve point (alpha_c = 0), gradient is zero (Eq. 4)
    if alpha_c < 1e-10:
        return np.zeros(d), svm
    
    # Identify margin support vectors S: 0 < alpha_i < C (Eq. 4)
    margin_sv_mask = (alpha_full > 1e-10) & (alpha_full < C_svm - 1e-10)
    margin_sv_indices = np.where(margin_sv_mask)[0]
    
    if len(margin_sv_indices) == 0:
        return np.zeros(d), svm
    
    # Build Q_ss: label-annotated kernel submatrix for margin SVs
    X_s = X_aug[margin_sv_indices]
    y_s = y_aug[margin_sv_indices]
    Q_ss = np.outer(y_s, y_s) * (X_s @ X_s.T)  # Q_ij = y_i * y_j * K(x_i, x_j)
    
    # Invert Q_ss (with Tikhonov regularisation for numerical stability)
    Q_ss_inv = np.linalg.inv(Q_ss + 1e-8 * np.eye(len(Q_ss)))
    
    # Compute upsilon and zeta from Eq. (8):
    # [0, y_s^T; y_s, Q_ss]^{-1} is decomposed using block inversion
    y_s_vec = y_s.reshape(-1, 1)
    upsilon = Q_ss_inv @ y_s_vec          # Eq. (8)
    zeta = float(y_s_vec.T @ upsilon)     # Eq. (8)
    
    if abs(zeta) < 1e-10:
        return np.zeros(d), svm
    
    # Compute validation margins g_k = y_k * f(x_k)
    f_val = svm.decision_function(X_val)
    g_val = y_val * f_val
    
    # Active set: validation points with hinge loss > 0 (i.e., g_k < 1)
    active_mask = g_val < 1.0
    
    if not np.any(active_mask):
        return np.zeros(d), svm
    
    # Accumulate gradient contributions from all active validation points
    gradient = np.zeros(d)
    
    for k in np.where(active_mask)[0]:
        x_k = X_val[k]
        y_k = y_val[k]
        
        # Q_ks = y_k * y_s * K(x_k, x_s) for linear kernel
        Q_ks = y_k * y_s * (x_k @ X_s.T)
        
        # M_k from Eq. (10):
        # M_k = -(1/zeta) * (Q_ks @ (zeta * Q_ss_inv - upsilon @ upsilon^T) + y_k * upsilon^T)
        M_k = -(1.0 / zeta) * (
            Q_ks @ (zeta * Q_ss_inv - upsilon @ upsilon.T) + y_k * upsilon.T
        )
        M_k = M_k.flatten()
        
        # For linear kernel: dQ_sc/dx_c = y_s * y_c * X_s, dQ_kc/dx_c = y_k * y_c * x_k
        # (Section 2.2: dK(x_i, x_c)/du = t * x_i for linear kernel)
        dQ_sc = (y_s * y_c).reshape(-1, 1) * X_s  # shape: (|S|, d)
        dQ_kc = y_k * y_c * x_k                    # shape: (d,)
        
        # Eq. (10): gradient contribution = (M_k @ dQ_sc + dQ_kc) * alpha_c
        gradient += (M_k @ dQ_sc + dQ_kc) * alpha_c
    
    return gradient, svm

The function `compute_poisoning_gradient_linear` implements the core mathematical contribution of the paper.
It computes $\partial L / \partial x_c$ (Equation 10) by:

1. Training the SVM on $D_{tr} \cup \{x_c, y_c\}$ and extracting dual variables $\alpha$ and bias $b$
2. Identifying margin support vectors ($S$: points with $0 < \alpha_i < C$, per Eq. 4)
3. Computing the inverse $Q_{ss}^{-1}$ and the Sherman–Morrison terms $\upsilon$, $\zeta$ (Eq. 8)
4. For each active validation point (hinge loss $> 0$), computing $M_k$ (Eq. 10) and accumulating the gradient

For the linear kernel, $\partial K(x_i, x_c) / \partial u = x_i$ (Section 2.2), simplifying the kernel gradient terms.

In [5]:
def poisoning_attack(X_train, y_train, X_val, y_val, X_test, y_test, C_svm,
                     n_iterations=100, step_size=0.02,
                     attacking_label=-1, random_seed=42):
    """
    Execute the gradient-based poisoning attack (Algorithm 1 of the paper).
    
    Steps:
    1. Initialise x_c by selecting a random attacked-class point and flipping its label
    2. At each iteration:
       a. Compute gradient dL/dx_c via compute_poisoning_gradient_linear
       b. Normalise to unit vector (attack direction u)
       c. Update x_c = x_c + step_size * u
       d. Clip to feature bounds [0, 1]
    3. Return the final attack point and error history
    
    Parameters
    ----------
    attacking_label : int — label assigned to the attack point (y_c)
    """
    np.random.seed(random_seed)
    
    attacked_label = -attacking_label
    
    # Step 1: Initialise by label-flipping (Algorithm 1, line 1-2)
    attacked_indices = np.where(y_train == attacked_label)[0]
    init_idx = np.random.choice(attacked_indices)
    x_c = X_train[init_idx].copy()
    y_c = attacking_label
    
    test_errors = []
    val_errors = []
    
    for iteration in range(n_iterations):
        # Step 2a: Compute gradient (Eq. 10)
        gradient, svm = compute_poisoning_gradient_linear(
            X_train, y_train, X_val, y_val, x_c, y_c, C_svm
        )
        
        # Record metrics
        X_aug = np.vstack([X_train, x_c.reshape(1, -1)])
        y_aug = np.append(y_train, y_c)
        svm_eval = SVC(kernel='linear', C=C_svm)
        svm_eval.fit(X_aug, y_aug)
        
        test_errors.append(1.0 - accuracy_score(y_test, svm_eval.predict(X_test)))
        val_errors.append(1.0 - accuracy_score(y_val, svm_eval.predict(X_val)))
        
        # Step 2b: Check convergence
        grad_norm = np.linalg.norm(gradient)
        if grad_norm < 1e-8:
            break
        
        # Step 2c: Normalise and update (Algorithm 1, lines 6-7)
        direction = gradient / grad_norm
        x_c = x_c + step_size * direction
        
        # Step 2d: Clip to bounds (Section 2.3: bounding box constraint)
        x_c = np.clip(x_c, 0, 1)
    
    return x_c, y_c, test_errors, val_errors

The `poisoning_attack` function implements **Algorithm 1** from the paper:

- **Initialisation (Line 1-2):** A random point from the attacked class is selected and its label is flipped
  to the attacking class. This ensures the initial point is already disruptive.
- **Gradient ascent loop (Lines 3-8):** At each iteration, we compute the gradient $\partial L / \partial u$,
  normalise it to a unit vector $u$, and take a step of size $t$ in that direction. The step size is kept
  small to satisfy the adiabatic condition (Eqs. 4-5), ensuring the SVM's support vector partition $\{S, E, R\}$
  remains stable. The bounding box constraint clips $x_c$ to $[0, 1]$, matching the paper's mention of
  bounds for linear kernels (Section 2.3).

In [6]:
# ── Run the single-point attack ──
print("Running single-point poisoning attack...")
x_poison, y_poison, test_errors, val_errors = poisoning_attack(
    X_train, y_train, X_val, y_val, X_test, y_test, C_SVM,
    n_iterations=150, step_size=0.02, random_seed=RANDOM_SEED
)

print(f"\nSingle-point attack results:")
print(f"  Clean test error:    {clean_test_err:.4f}")
print(f"  Poisoned test error: {test_errors[-1]:.4f}")
print(f"  Error increase:      {test_errors[-1] - clean_test_err:.4f}")

Running single-point poisoning attack...

Single-point attack results:
  Clean test error:    0.0400
  Poisoned test error: 0.0450
  Error increase:      0.0050


The single-point attack is executed with 150 iterations and step size $t = 0.02$. The error increase
from a single poisoned point demonstrates the concept, though the magnitude is modest because the
Breast Cancer dataset is well-separated in 30 dimensions. The paper reports much larger increases
(2-5% → 15-20%) on MNIST digit pairs (Section 3.2), where the classes overlap more and the
attack has more room to distort the decision boundary.

In [8]:
# ── Run multi-point sequential attack (replicating Figure 3 of the paper) ──
print("\nRunning multi-point sequential attack...")
percentages = [1, 2, 3, 4, 5, 6, 7, 8]
multi_test_errors_mean = []
multi_val_errors_mean = []

N_RUNS = 3  # Average over multiple random seeds

for pct in percentages:
    n_attack = max(1, int(pct * len(X_train) / 100))
    run_test_errors = []
    run_val_errors = []
    
    for run in range(N_RUNS):
        X_multi = X_train.copy()
        y_multi = y_train.copy()
        
        for i in range(n_attack):
            x_p, y_p, _, _ = poisoning_attack(
                X_multi, y_multi, X_val, y_val, X_test, y_test, C_SVM,
                n_iterations=80, step_size=0.02, random_seed=RANDOM_SEED + run * 100 + i
            )
            X_multi = np.vstack([X_multi, x_p.reshape(1, -1)])
            y_multi = np.append(y_multi, y_p)
        
        svm_m = SVC(kernel='linear', C=C_SVM)
        svm_m.fit(X_multi, y_multi)
        run_test_errors.append(1.0 - accuracy_score(y_test, svm_m.predict(X_test)))
        run_val_errors.append(1.0 - accuracy_score(y_val, svm_m.predict(X_val)))
    
    multi_test_errors_mean.append(np.mean(run_test_errors))
    multi_val_errors_mean.append(np.mean(run_val_errors))
    print(f"  {pct}% ({n_attack} pts): test_err={multi_test_errors_mean[-1]:.4f}, val_err={multi_val_errors_mean[-1]:.4f}")


Running multi-point sequential attack...
  1% (1 pts): test_err=0.0400, val_err=0.0267
  2% (2 pts): test_err=0.0450, val_err=0.0267
  3% (3 pts): test_err=0.0417, val_err=0.0267
  4% (4 pts): test_err=0.0433, val_err=0.0267
  5% (5 pts): test_err=0.0433, val_err=0.0233
  6% (6 pts): test_err=0.0433, val_err=0.0200
  7% (7 pts): test_err=0.0433, val_err=0.0233
  8% (8 pts): test_err=0.0433, val_err=0.0267


The multi-point attack replicates the experimental setup from **Figure 3** of the paper, where
attack points are sequentially injected and the resulting classification error is measured as a
function of the percentage of poisoned points in the training data. Results are averaged over
3 random seeds to reduce variance (the paper also averages over multiple runs with randomly
chosen training/validation sets).

In [9]:
# ── Plot: Single-point error progression ──
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(test_errors)), test_errors, label='Test Error', color='#1565C0', linewidth=2)
ax.plot(range(len(val_errors)), val_errors, label='Validation Error', color='#C62828', linewidth=2, linestyle='--')
ax.axhline(y=clean_test_err, color='#757575', linestyle=':', label=f'Clean Error ({clean_test_err:.4f})', linewidth=1.5)
ax.set_xlabel('Attack Iteration', fontsize=12)
ax.set_ylabel('Classification Error', fontsize=12)
ax.set_title('Single-Point Poisoning Attack: Error Over Iterations\n(Breast Cancer Dataset, Linear Kernel, C=1)', fontsize=13)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/yashlunawat/C/sem6/AML/230110-midsem/partB/results/task2_single_point_error.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: partB/results/task2_single_point_error.png")

Saved: partB/results/task2_single_point_error.png


This plot shows how the test and validation errors evolve as the attack point $x_c$ is optimised
across iterations. It is analogous to the rightmost column of **Figure 2** in the paper, which
shows the increase in validation and testing errors across attack iterations for MNIST digit pairs.

In [10]:
# ── Plot: Multi-point attack (analogous to Figure 3) ──
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.plot(percentages, multi_test_errors_mean, 'o-', label='Test Error', color='#1565C0', linewidth=2, markersize=8)
ax2.plot(percentages, multi_val_errors_mean, 's--', label='Validation Error', color='#C62828', linewidth=2, markersize=8)
ax2.axhline(y=clean_test_err, color='#757575', linestyle=':', label=f'Clean Error ({clean_test_err:.4f})', linewidth=1.5)
ax2.set_xlabel('% of Attack Points in Training Data', fontsize=12)
ax2.set_ylabel('Classification Error', fontsize=12)
ax2.set_title('Multi-Point Poisoning Attack\n(Breast Cancer Dataset, Linear Kernel, C=1)', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/yashlunawat/C/sem6/AML/230110-midsem/partB/results/task2_multi_point_attack.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: partB/results/task2_multi_point_attack.png")

Saved: partB/results/task2_multi_point_attack.png


This plot replicates the format of **Figure 3** from the paper, showing how classification error
grows as the percentage of poisoned points in the training data increases. The paper observes
a "steady growth of the attack effectiveness with the increasing percentage of the attack points
in the training set" (Section 3.2), which our results confirm on the Breast Cancer dataset.